# Group 1 stock factor regressions

Apply the six classroom steps to the stocks in stagger_1 only. See `README.md` for methodology and assumptions.

## Run the analysis, then walk through the six classroom steps

The next cell computes the complete pipeline. The walkthrough then examines each stage in order using recorded intermediate standard deviations and coefficients. Foreign stocks follow steps 1 → 2 → 4 → 5 → 6; US stocks follow steps 3 → 4 → 5 → 6. Every comparison uses the same observations before and after the stage for a given stock.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().parent if Path.cwd().name == "group1" else Path.cwd()
sys.path.insert(0, str(root / "group1"))
from stock_factor_regressions import run

tables, residuals = run()

## Visual guide to the regression results

Run the cells above first. These charts use the current in-memory `tables` and `residuals` for **stagger_1**. Betas describe sequential, order-dependent regressions; sector betas refer to sector residual factors. They are not coefficients from a single simultaneous regression.

Returns already include the class time weights. Residual standard deviations below are in those weighted four-week return units, not annualized volatility. Selection is exploratory and in-sample.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False})
plot_stagger = "stagger_1"
plot_diagnostics = (tables["diagnostics"].query("stagger == @plot_stagger")
                    .sort_values(["primary_sector", "ticker"]).set_index("ticker"))
plot_order = plot_diagnostics.index
plot_exposures = (tables["factor_exposures"].query("stagger == @plot_stagger")
                  .set_index("ticker").loc[plot_order])
plot_stages = tables["stage_coefficients"].query("stagger == @plot_stagger")
plot_residuals = residuals[plot_stagger].reindex(columns=plot_order)
plot_labels = [f"{ticker} ({plot_diagnostics.loc[ticker, 'primary_sector']})"
               for ticker in plot_order]

In [ ]:
from IPython.display import display

walk_progress = (tables["residual_progress"].query("stagger == @plot_stagger")
                 .pivot(index="ticker", columns="stage", values="residual_std"))

def show_stage(step, before, after, foreign=None):
    tickers = plot_order
    if foreign is not None:
        mask = plot_diagnostics["currency"].ne("USD")
        tickers = plot_diagnostics.index[mask if foreign else ~mask]
    comparison = walk_progress.loc[tickers, [before, after]].copy()
    comparison.columns = ["Before std", "After std"]
    comparison["Reduction (%)"] = 100 * (1 - comparison["After std"] /
                                        comparison["Before std"].replace(0, np.nan))
    comparison["Observations"] = plot_diagnostics.loc[tickers, "observations"]
    coefficients = plot_stages.loc[plot_stages["step"].eq(step) & plot_stages["ticker"].isin(tickers),
                                  ["ticker", "factor", "beta", "applied_beta", "t_stat", "retained"]]
    display(coefficients.reset_index(drop=True))
    if comparison.empty:
        print("No stocks follow this branch of the workflow.")
        return
    display(comparison.round(4))
    fig, ax = plt.subplots(figsize=(11, max(4, len(tickers) * 0.3)), layout="constrained")
    positions = np.arange(len(tickers))
    ax.barh(positions + 0.18, comparison["Before std"], height=0.36, color="#B8BDC5", label=before)
    ax.barh(positions - 0.18, comparison["After std"], height=0.36, color="#2878A5", label=after)
    ax.set_yticks(positions, tickers, fontsize=8)
    ax.set(title=f"Step {step}: residual variation before and after",
           xlabel="Sample std (weighted four-week return units)")
    ax.legend()
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)
    plt.show()

## Step 1 — Foreign stocks: home-currency exposure

Regress each foreign stock on its home currency with an intercept. Keep the estimated beta and pass the residuals to step 2. US stocks skip this step.

In [ ]:
show_stage(1, 'Original', 'Home FX', foreign=True)

## Step 2 — Foreign stocks: test AUD exposure

Regress the step-1 residuals on AUD/USD. Keep the fitted component only if |t| > 2; otherwise apply beta = 0 and pass the response through unchanged. `beta` is the tested estimate; `applied_beta` records the decision.

In [ ]:
show_stage(2, 'Home FX', 'AUD', foreign=True)

## Step 3 — US stocks: test AUD exposure

This is the alternative to steps 1–2 for US stocks. Test AUD/USD against the original weighted return using the same strict |t| > 2 rule. Rejected fits leave the response unchanged.

In [ ]:
show_stage(3, 'Original', 'AUD', foreign=False)

## Step 4 — All stocks: remove SPY exposure

Regress the response from the applicable AUD stage on SPY with an intercept. Retain the market beta and pass the residuals forward.

In [ ]:
show_stage(4, 'AUD', 'SPY', foreign=None)

## Step 5 — All stocks: remove the primary sector factor

Regress step-4 residuals on the stock’s primary sector residual factor. Always retain this beta and pass the new residuals forward. Sector factors are already residualized against SPY in the source workbook.

In [ ]:
show_stage(5, 'SPY', 'Primary sector', foreign=None)

## Step 6 — All stocks: select additional sector factors

Forward-select remaining sector residual factors against the fixed step-5 response. Each addition must leave every selected coefficient at |t| > 2 in the joint refit. The coefficient table shows the final retained set; stocks with no qualifying additions stay unchanged. The chart compares step 5 with the final joint fit.

In [ ]:
show_stage(6, 'Primary sector', 'Additional sectors', foreign=None)

### Are residuals getting smaller at each stage?

Each line tracks one stock on its fixed complete-case sample. Standard deviations are normalized to that stock's original standard deviation: 1 means no reduction, 0.5 means half the original variation. Retained OLS stages with an intercept cannot increase sample residual variance, apart from numerical tolerance; skipped stages stay flat. The additional-sector point summarizes the final joint fit, not every candidate search iteration.

The heatmap shows the **percentage reduction in standard deviation relative to the preceding stage**. Zero can mean a skipped stage or negligible improvement. This is an in-sample fit check, not evidence of out-of-sample predictive power. The table reports any increases exceeding numerical tolerance.

In [ ]:
stage_order = ["Original", "Home FX", "AUD", "SPY", "Primary sector", "Additional sectors"]
progress = (tables["residual_progress"].query("stagger == @plot_stagger")
            .pivot(index="ticker", columns="stage", values="residual_std")
            .reindex(index=plot_order, columns=stage_order))
# A zero starting standard deviation has no defined proportional reduction.
normalized = progress.div(progress["Original"].replace(0, np.nan), axis=0)
previous_std = progress.shift(axis=1)
step_reduction = 100 * (1 - progress.div(previous_std.replace(0, np.nan)))
step_reduction = step_reduction.iloc[:, 1:]
increase_tolerance = 1e-12 + 1e-10 * previous_std.abs()
increases = (progress - previous_std) > increase_tolerance
violations = (progress - previous_std).where(increases).stack().dropna().rename("std_increase")
print(f"Checked {len(progress)} stocks across {len(stage_order) - 1} transitions: "
      f"{len(violations)} increases beyond numerical tolerance.")
if not violations.empty:
    display(violations.to_frame())

fig, axes = plt.subplots(1, 2, figsize=(16, max(9, len(progress) * 0.27)),
                         layout="constrained", gridspec_kw={"width_ratios": [1.2, 1]})
for ticker, row in normalized.iterrows():
    axes[0].plot(range(len(stage_order)), row, color="#2878A5", alpha=0.22, linewidth=1)
axes[0].plot(range(len(stage_order)), normalized.median(), color="#172D46",
             marker="o", linewidth=3, label="Median across stocks")
axes[0].set_xticks(range(len(stage_order)), stage_order, rotation=35, ha="right")
axes[0].set(title="Does residual variation fall at each stage?",
            ylabel="Residual std / original std", ylim=(0, max(1.05, normalized.max().max() * 1.05)))
axes[0].grid(alpha=0.2)
axes[0].legend()

cmap = plt.get_cmap("RdBu").copy()
cmap.set_bad("#DDDDDD")
limit = max(1.0, float(np.nanmax(np.abs(step_reduction.to_numpy()))))
im = axes[1].imshow(step_reduction, cmap=cmap, vmin=-limit, vmax=limit, aspect="auto")
axes[1].set_xticks(range(len(stage_order) - 1), stage_order[1:], rotation=35, ha="right")
axes[1].set_yticks(range(len(progress)), plot_order, fontsize=8)
axes[1].set_title("Reduction from preceding stage (%)")
for row in range(len(progress)):
    for col in range(len(stage_order) - 1):
        value = step_reduction.iloc[row, col]
        if np.isfinite(value):
            axes[1].text(col, row, f"{value:.1f}", ha="center", va="center", fontsize=7,
                         color="white" if abs(value) > limit * 0.6 else "#222222")
fig.colorbar(im, ax=axes[1], shrink=0.55, label="Std reduction (%) — negative means increase")
plt.show()

## Overall factor and residual diagnostics

### Which factors drive each stock?

Rows are grouped by primary sector (shown in parentheses). Red indicates positive exposure; blue indicates negative exposure. Currency and equity panels have separate symmetric color scales so large FX betas do not hide equity exposures. White means zero applied exposure, including skipped or rejected factors; it does not establish that the true exposure is zero.

In [ ]:
fx_columns = [c for c in plot_exposures if c.startswith("beta_") and "/USD" in c]
equity_columns = [c for c in plot_exposures if c.startswith("beta_") and "/USD" not in c]
fig, axes = plt.subplots(1, 2, figsize=(16, max(8, len(plot_order) * 0.27)),
                         sharey=True, layout="constrained",
                         gridspec_kw={"width_ratios": [len(fx_columns), len(equity_columns)]})
for ax, columns, title in zip(axes, [fx_columns, equity_columns],
                             ["Currency exposures", "SPY and sector-residual exposures"]):
    values = plot_exposures[columns].to_numpy(dtype=float)
    limit = max(float(np.nanmax(np.abs(values))), 0.01)
    im = ax.imshow(values, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    ax.set_xticks(range(len(columns)), [c.removeprefix("beta_") for c in columns],
                  rotation=45, ha="right")
    ax.set_yticks(range(len(plot_order)), plot_labels, fontsize=8)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.55, label="Applied beta")
axes[0].set_ylabel("Stock (primary sector)")
fig.suptitle(f"Factor exposure map — {plot_stagger}", fontsize=15)
plt.show()

### How much market exposure remains after the currency stages?

SPY betas are estimated after any retained currency stages and before the sector stages. The dashed line marks beta = 1; the solid line marks zero. Negative values indicate an inverse relationship at this stage. This ranking shows estimates, not their statistical significance.

In [ ]:
spy_beta = plot_exposures["beta_SPY"].sort_values()
fig, ax = plt.subplots(figsize=(10, max(7, len(spy_beta) * 0.25)), layout="constrained")
ax.barh(spy_beta.index, spy_beta, color=np.where(spy_beta >= 0, "#2878A5", "#C65C46"))
ax.axvline(0, color="#333333", linewidth=0.8)
ax.axvline(1, color="#777777", linestyle="--", linewidth=1, label="Beta = 1")
ax.set(title="SPY exposure after currency adjustments", xlabel="Applied SPY beta", ylabel="Stock")
ax.tick_params(axis="y", labelsize=8)
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)
plt.show()

### Which stocks pass the AUD selection rule?

The bars show the **tested** AUD t-statistic, including rejected estimates. AUD is retained only when |t| > 2; the dashed lines show the strict cutoff. For foreign stocks this test follows the home-currency regression; for US stocks it uses the original weighted return. Classical OLS t-statistics are not adjusted for multiple testing.

In [ ]:
aud = plot_stages.loc[plot_stages["factor"].eq("AUD/USD")].sort_values("t_stat")
fig, ax = plt.subplots(figsize=(10, max(7, len(aud) * 0.25)), layout="constrained")
for retained, color, label in [(True, "#2878A5", "Retained"), (False, "#B8BDC5", "Rejected")]:
    subset = aud.loc[aud["retained"].eq(retained)]
    positions = np.flatnonzero(aud["retained"].eq(retained).to_numpy())
    ax.barh(positions, subset["t_stat"], color=color, label=label)
ax.set_yticks(range(len(aud)), aud["ticker"], fontsize=8)
for cutoff in [-2, 2]:
    ax.axvline(cutoff, color="#C65C46", linestyle="--", linewidth=1)
ax.axvline(0, color="#333333", linewidth=0.8)
ax.set(title="AUD/USD: tested t-statistics and applied selection", xlabel="AUD coefficient t-statistic",
       ylabel="Stock")
ax.legend()
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)
plt.show()

### Where is the largest remaining variation?

Stocks are ranked by their final residual sample standard deviation (ddof = 1). Labels report the number of observations used for each stock, since shorter listing histories can make comparisons less reliable. Values are neither annualized nor unweighted stock volatility.

In [ ]:
risk = plot_diagnostics.sort_values("residual_std")
fig, ax = plt.subplots(figsize=(10, max(7, len(risk) * 0.25)), layout="constrained")
ax.barh(range(len(risk)), risk["residual_std"], color="#37877C")
ax.set_yticks(range(len(risk)),
              [f"{ticker} (n={int(n)})" for ticker, n in risk["observations"].items()], fontsize=8)
ax.set(title="Remaining variation after all retained factors",
       xlabel="Final residual standard deviation (weighted four-week return units)", ylabel="Stock")
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)
plt.show()

### Do stocks still move together after the factor stages?

This correlation map uses only dates with final residuals for **every** stock, giving every pair the same sample. The title reports that sample size; no missing returns are filled. Stocks follow the same sector grouping as the exposure map. Strong off-diagonal patterns suggest remaining shared variation, not proof of an omitted causal factor or independent residuals. Constant residual series yield undefined (gray) correlations.

In [ ]:
common_residuals = plot_residuals.dropna(how="any")
if len(common_residuals) < 3:
    print("Too few shared dates to plot residual correlations (need at least 3).")
else:
    correlation = common_residuals.corr()
    fig, ax = plt.subplots(figsize=(13, 12), layout="constrained")
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad("#DDDDDD")
    im = ax.imshow(correlation, vmin=-1, vmax=1, cmap=cmap)
    ax.set_xticks(range(len(plot_order)), plot_order, rotation=90, fontsize=8)
    ax.set_yticks(range(len(plot_order)), plot_labels, fontsize=8)
    ax.set_title(f"Final residual correlations — {len(common_residuals)} shared dates", pad=15)
    fig.colorbar(im, ax=ax, shrink=0.75, label="Pearson correlation")
    plt.show()

## Full output tables

In [ ]:
tables["diagnostics"]

In [ ]:
tables["factor_exposures"]

In [ ]:
tables["stage_coefficients"].query("factor == 'AUD/USD'")

In [ ]:
residuals["stagger_1"]